# Info

This notebook:
- downloads data from Firebase
- create Hugging Face datasets for Whisper training

## Requirements

Run this notebook using a kernel which has the `ipykernel` module.

To install the module:
- Open a terminal
- [if a venv is used, source it]
- Install the `ipykernel` module:
```bashrc
pip install ipykernel
```

# Set parameters, hyper-parameters

In [ ]:
# eval dataset on hugging face

# If set, the Hugging Face dataset repo must already exist (otherwise this notebook will raise an error).
# If that repo contains a data.json file, it will be used as the starting point to grow the eval split.
# If this is set to None, the notebook will create an eval split from scratch.
hf_repo_dataset_eval = "luca-r/chiara_dataset-eval_20261104"

In [ ]:
# Credentials

# env file with configs and credentials
DIR_ENV_FILE            = "/path/to/chiara-speech2text_envs/.env"

In [ ]:
# Directories for downloaded data

# Directory for raw  data
DIR_DATA_RAW            = "/path/to/chiara-speech2text_data/1-raw"

# Directory for processed data
DIR_DATA_PROCESSED      = "/path/to/chiara-speech2text_data/temp/2-datasets"

# Directory for hugging face datasets
DIR_DATA_HF             = "/path/to/chiara-speech2text_data/temp/3-hf_datasets"

In [ ]:
# Download folders

# parameter_download_data_from_firebase = False
parameter_download_data_from_firebase = True

In [ ]:
# choose which datasets to generate
parameter_generate_dataset_sft   = False
parameter_generate_dataset_kto   = True
parameter_generate_dataset_ipo   = False

# ratio of samples of all data which will be devoted to the evaluation dataset
# (the rest is used for training + testing)
parameter_ratio_eval              = 0.1       # 0.1 = 10% for eval split (remaining 90% for training + testing)

# ratio of data left-aside from eval which will be used for training
# (the rest is used for testing)
parameter_ratio_train             = 0.9       # 0.9 = 90% for train split (remaining 10% for testing)

# for samples with reviewer_is_reviewed=False and user_is_validated=True
# get only samples with inference_confidence >= threshold
#parameter_confidence_threshold_pct    = 50    # 50 = 50% confidence threshold
parameter_confidence_threshold_pct    = 0

In [ ]:
# feature extractor

# this is used by hf_processor to extract features from audio files and to tokenize text labels
# NOTE: you must use the same base Whisper base-model that will be used as base-model for the fine-tuned model
# (PS: no need to use the same fine-tuned model, just the same base-model as the fine-tuned model)

parameter_base_model = "openai/whisper-large-v3"
#parameter_base_model = "openai/whisper-large-v3-turbo"

# ToDo

In [ ]:
# ...

# Init notebook

In [ ]:
# Import utils

import os, sys
from pathlib import Path

repo_root = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "utils" / "utils.py").exists():
        repo_root = candidate
        break

if repo_root is None:               raise FileNotFoundError("Could not find the repository root containing utils/utils.py")
if str(repo_root) not in sys.path:  sys.path.insert(0, str(repo_root))

from utils.utils import (
    download_firebase_folders,
    create_dataset_eval,
    create_dataset_sft,
    create_dataset_kto,
    create_dataset_ipo,
    compare_dataset_overlaps,
)

# Use the currently active notebook environment (the existing venv/kernel)
print(f"Python executable: {sys.executable}")
print(f"Virtual env:       {os.environ.get('VIRTUAL_ENV', 'not set')}")
print(f"Repository root:   {repo_root}")

try:    del candidate, repo_root
except: pass

In [ ]:
# Install packages
# This cell requires some time on the 1st run (~8m30s).
# Remove the '--quiet' option to check the installation progress

%pip install --quiet gsutil
%pip install --quiet dotenv
%pip install --quiet datasets
%pip install --quiet ipywidgets
%pip install --quiet librosa
%pip install --quiet matplotlib
%pip install --quiet transformers
#%pip install --quiet tensorflow
%pip install --quiet torch

# numpy <2.4 for numba compatibility
%pip uninstall --quiet -y numpy
%pip install --quiet "numpy<=2.4.0"

In [ ]:
# Import variables from .env file

# Load the variables from the .env file into the environment
from dotenv import load_dotenv
load_dotenv(DIR_ENV_FILE)

In [ ]:
# get HF env variables

ENV_HF_USER     = os.getenv("HF_USER")
ENV_HF_TOKEN    = os.getenv("HF_TOKEN")

In [ ]:
# get Firebase env variables

DIR_FIREBASE_BUCKET = os.environ.get("TRAINER_FIREBASE_BUCKET")

In [ ]:
# Disable ipywidgets bars (they cause visualization errors with transforms library in notebooks)

import os
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"]      = "1"

import datasets
datasets.disable_progress_bars()

# Manage data

## Download from Firebase bucket

In [ ]:
# Create folders

# DIR_DATA_RAW
# do NOT delete the raw data folder, as it contains the data previoulsy (already) downloaded from Firebase
# it takes time to download data from Firebase, so we want to keep it for future runs of this notebook
!mkdir -p {DIR_DATA_RAW}

In [ ]:
# Download folders

if parameter_download_data_from_firebase:
    # download
    folders_downloaded = download_firebase_folders(
        folders                     = None,                 # None: download all folders from the Firebase bucket
        dir_firebase_bucket         = DIR_FIREBASE_BUCKET,
        dir_output                  = DIR_DATA_RAW,
        force_download_data_json    = True,                 # True: force re-download of jsons (to get latest text-edits  done in frontend)
        force_download_wav          = False                 # True: force re-download of wavs  (to get latest audio-trims done in frontend)
    )

    # print
    print("The following folders have been downloaded:")
    for folder in folders_downloaded:
        print(f"   - {folder}")

else:
    print("'parameter_download_data_from_firebase' has been set to False -> no data will be downloaded")

# Create datasets

In [ ]:
# DIR_DATA_PROCESSED
#!rm -rf     {DIR_DATA_PROCESSED} # remove old folder, to create fresh datasets
!mkdir -p   {DIR_DATA_PROCESSED}

In [ ]:
# dataset-eval json

from pathlib import Path
import shutil
from huggingface_hub import HfApi, hf_hub_download

has_eval_dataset_json       = False
existing_eval_dataset_path  = None
   
# check if hf_repo_dataset_eval is set, and whether it already contains a dataset.json file
if hf_repo_dataset_eval:
    print(f"Checking if Hugging Face dataset repo '{hf_repo_dataset_eval}' exists...")
    if not HfApi().repo_exists(repo_id=hf_repo_dataset_eval, repo_type="dataset"):
        raise FileNotFoundError(
            f"Hugging Face dataset repo '{hf_repo_dataset_eval}' was not found. "
            "Set hf_repo_dataset_eval to None to create a new eval split from scratch, "
            "or provide an existing dataset repo ID."
        )

    print(f"Checking if eval dataset.json exists in '{hf_repo_dataset_eval}'...")

    try:
        # check if "dataset.json" exists in the Hugging Face dataset repo
        has_eval_dataset_json = HfApi().file_exists(
            repo_id     = hf_repo_dataset_eval,
            filename    = "data.json",
            repo_type   = "dataset",
        )

        # download existing dataset-eval.json if it exists
        if has_eval_dataset_json:
            dir_eval = Path(DIR_DATA_PROCESSED)
            dir_eval.mkdir(parents=True, exist_ok=True)
            existing_eval_dataset_path = dir_eval / "dataset-eval.json"
            downloaded_eval_dataset_path = hf_hub_download(
                repo_id     = hf_repo_dataset_eval,
                filename    = "data.json",
                repo_type   = "dataset",
            )
            shutil.copy2(downloaded_eval_dataset_path, existing_eval_dataset_path)
            print(f"Downloaded existing data.json from '{hf_repo_dataset_eval}' to '{existing_eval_dataset_path}'")

    except Exception as exc:
        raise RuntimeError(f"Could not check dataset.json in '{hf_repo_dataset_eval}': {exc}") from exc

else:
    print("No Hugging Face eval dataset repo ID was provided. A new dataset-eval will be created from scratch.")

#
if has_eval_dataset_json:   print(f"Found eval data.json in '{hf_repo_dataset_eval}'. This will be used as starting point for dataset-eval")
else:                       print(f"Eval data.json was not found in '{hf_repo_dataset_eval}'. A new dataset-eval will be created from scratch")
        
#
dir_eval = Path(DIR_DATA_PROCESSED)
dir_eval.mkdir(parents=True, exist_ok=True)

#
result_eval = create_dataset_eval(
    folders                             = None,                         # None: use all folders available in DIR_DATA_RAW
    dir_input                           = DIR_DATA_RAW,
    dir_output                          = str(dir_eval),
    ratio                               = parameter_ratio_eval,
    #ground_truth_source                = "only_reviewer",             # only get entries with reviewer_is_reviewed=True
    #ground_truth_source                = "only_user",                 # only get entries with user_is_validated=True
    ground_truth_source                 = "reviewer_and_user",         # get entries with EITHER reviewer_is_reviewed=True OR user_is_validated=True
    inference_confidence_threshold_pct  = parameter_confidence_threshold_pct,
    existing_eval_dataset_path          = existing_eval_dataset_path,
    output_filename                     = "dataset-eval.json"
)

dataset_eval = result_eval["json_path"]
print("dataset-eval json file:", dataset_eval)


# cleanup
del dir_eval, existing_eval_dataset_path, result_eval

In [ ]:
# dataset-sft

if parameter_generate_dataset_sft:
    from pathlib import Path

    #
    print("I will generate dataset-sft")
    dir_sft = Path(DIR_DATA_PROCESSED)
    dir_sft.mkdir(parents=True, exist_ok=True)

    #
    result_sft = create_dataset_sft(
        folders                             = None,                                 # use all folders available in DIR_DATA_RAW
        dir_input                           = DIR_DATA_RAW,
        dir_output                          = str(dir_sft),
        ground_truth_source                 = "reviewer_and_user",                  # get entries with EITHER reviewer_is_reviewed=True OR user_is_validated=True
        inference_confidence_threshold_pct  = parameter_confidence_threshold_pct,
        exclude_entries                     = dataset_eval,                         # exclude entries in the evaluation dataset
        output_filename                     = "dataset-sft.json"
    )

    #
    dataset_sft = result_sft["json_path"]
    print("dataset-sft json file:", dataset_sft)

    # cleanup
    del dir_sft, result_sft

In [ ]:
# dataset-kto

if parameter_generate_dataset_kto:
    from pathlib import Path

    #
    print("I will generate dataset-kto")
    dir_kto = Path(DIR_DATA_PROCESSED)
    dir_kto.mkdir(parents=True, exist_ok=True)

    #
    result_kto = create_dataset_kto(
        folders                             = None,                                 # use all folders available in DIR_DATA_RAW
        dir_input                           = DIR_DATA_RAW,
        dir_output                          = str(dir_kto),
        ground_truth_source                 = "reviewer_and_user",                  # get entries with EITHER reviewer_is_reviewed=True OR user_is_validated=True
        inference_confidence_threshold_pct  = parameter_confidence_threshold_pct,
        exclude_entries                     = dataset_eval,                         # exclude entries in the evaluation dataset
        output_filename                     = "dataset-kto.json"
    )

    #
    dataset_kto = result_kto["json_path"]
    print("dataset-kto json file:", dataset_kto)

    # cleanup
    del dir_kto, result_kto

In [ ]:
# dataset-ipo

if parameter_generate_dataset_ipo:
    from pathlib import Path

    #
    print("I will generate dataset-ipo")
    dir_ipo = Path(DIR_DATA_PROCESSED)
    dir_ipo.mkdir(parents=True, exist_ok=True)

    #
    result_ipo = create_dataset_ipo(
        folders                             = None,                                 # use all folders available in DIR_DATA_RAW
        dir_input                           = DIR_DATA_RAW,
        dir_output                          = DIR_DATA_PROCESSED,
        transcribe_endpoint                 = "http://127.0.0.1:8000",              # endpoint of the transcribe service
        exclude_entries                     = dataset_eval,                         # exclude entries in the evaluation dataset
        output_filename                     = "dataset-ipo.json",
        max_entries                         = 10                                    # Only process first 10 entries, for testing purposes
    )

    #
    dataset_ipo = result_ipo["json_path"]
    print("dataset-ipo json file:", dataset_ipo)

    # cleanup
    del dir_ipo, result_ipo

## Check for overlaps

In [ ]:
# dataset_eval <> dataset_sft
if parameter_generate_dataset_sft:
    overlap_result = compare_dataset_overlaps(dataset_eval, dataset_sft)
    
    if overlap_result["has_overlaps"]:
        raise RuntimeError(
            f"❌ Found {overlap_result['overlap_count']} overlapping audio file(s) "
            "between eval and sft datasets."
        )
    else:
        print("✅ No overlaps found between eval and sft datasets.")

# dataset_eval <> dataset_kto
if parameter_generate_dataset_kto:
    overlap_result = compare_dataset_overlaps(dataset_eval, dataset_kto)
    
    if overlap_result["has_overlaps"]:
        raise RuntimeError(
            f"❌ Found {overlap_result['overlap_count']} overlapping audio file(s) "
            "between eval and kto datasets."
        )
    else:
        print("✅ No overlaps found between eval and kto datasets.")

# dataset_sft <> dataset_ipo
if parameter_generate_dataset_ipo:
    overlap_result = compare_dataset_overlaps(dataset_eval, dataset_ipo)
    
    if overlap_result["has_overlaps"]:
        raise RuntimeError(
            f"❌ Found {overlap_result['overlap_count']} overlapping audio file(s) "
            "between sft and ipo datasets."
        )
    else:
        print("✅ No overlaps found between eval and ipo datasets")

## Normalize texts

In [ ]:
from pathlib import Path
import json
from utils.utils import my_normalizer

field_map           = {}
substitutions_files = {}
#
if parameter_generate_dataset_sft:
    field_map["dataset-sft.json"]              = ["ground_truth"]                                           # this is the property that contains the text to be normalized
    substitutions_files["dataset-sft.json"]    = f"{DIR_DATA_PROCESSED}/dataset-sft_normalizations.json"    # this file will contain all substitutions done during normalization
#
if parameter_generate_dataset_kto:
    field_map["dataset-kto.json"]              = ["ground_truth"]                                           # this is the property that contains the text to be normalized
    substitutions_files["dataset-kto.json"]    = f"{DIR_DATA_PROCESSED}/dataset-kto_normalizations.json"    # this file will contain all substitutions done during normalization
#
if parameter_generate_dataset_ipo:
    field_map["dataset-ipo.json"]              = ["ipo_transcript_winner", "ipo_transcript_loser"]                                           # this is the property that contains the text to be normalized
    substitutions_files["dataset-ipo.json"]    = f"{DIR_DATA_PROCESSED}/dataset-ipo_normalizations.json"    # this file will contain all substitutions done during normalization
#

#
for file_name, fields in field_map.items():
    file_path = f"{DIR_DATA_PROCESSED}/{file_name}"
    if not Path(file_path).exists():
        print(f"Skipping missing file: {file_path}")
        continue

    print(f"Normalizing {file_path}...")
    with Path(file_path).open("r", encoding="utf-8") as f:
        rows = json.load(f)

    substitutions   = []
    changed         = False

    for row in rows:
        for field in fields:
            value = row.get(field)
            if not isinstance(value, str) or not value.strip():
                continue

            normalized = my_normalizer(
                value,
                audio_filename          = row.get("audio_data"),
                substitutions_log_path  = None,
                verbose                 = True,
            )
            if normalized is not None and normalized != value:
                substitutions.append(
                    {
                        "     audio_filename": row.get("audio_data"),
                        "normalized_property": field,
                        "      original_text": value,
                        "    normalized_text": normalized,
                    }
                )
                row[field] = normalized
                changed = True

    substitutions_path = substitutions_files[file_name]
    with Path(substitutions_path).open("w", encoding="utf-8") as f:
        json.dump(substitutions, f, ensure_ascii=False, indent=2)

    if changed:
        with Path(file_path).open("w", encoding="utf-8") as f:
            json.dump(rows, f, ensure_ascii=False, indent=2)
        print(f"  Saved normalized values to {file_path}")
        print(f"  Stored all substitutions performed to {substitutions_path}")
    else:
        print(f"  No changes needed for {file_path}")

# cleanup
try:
    del f, field, field_map, substitutions_files
    del file_name, fields, file_path, rows, substitutions, changed
    del value, normalized, substitutions_path
    del row
except: pass

## Hugging Face datasets

In [ ]:
# DIR_DATA_HF

!rm -rf {DIR_DATA_HF}
!mkdir -p {DIR_DATA_HF}

In [ ]:
# Helpers: load JSON and create HF DatasetDict splits

from pathlib import Path
import json
import random
from datasets import Dataset, DatasetDict

def _load_json(path):
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"JSON file not found: {p}")
    with p.open('r', encoding='utf-8') as fh:
        data = json.load(fh)
    # Handle common wrappers
    if isinstance(data, dict) and 'data' in data:
        data = data['data']
    if not isinstance(data, list):
        raise TypeError(f"Expected a list of entries in {p}, got {type(data).__name__}")
    return data

def create_hf_dataset_from_json(json_path, text_fields, label_fn=None, train_ratio=0.8, shuffle_seed=42, split_name=None):
    rows = _load_json(json_path)
    mapped = []
    for r in rows:
        # try common audio field names
        audio = r.get('audio_data') or r.get('audio_path') or r.get('audio')
        # pick first available text field
        text = None
        for f in text_fields:
            val = r.get(f)
            if isinstance(val, str) and val.strip():
                text = val.strip()
                break
        if not text:
            continue
        out = {
            'audio': str(audio) if audio else None,
            'text': text,
        }
        if label_fn is not None:
            try:
                out['label'] = label_fn(r)
            except Exception:
                out['label'] = None
        mapped.append(out)

    rng = random.Random(shuffle_seed)
    rng.shuffle(mapped)
    full_dataset    = Dataset.from_list(mapped)
    if split_name is not None:
        return DatasetDict({split_name: full_dataset})

    split           = int(len(mapped) * train_ratio)
    train           = full_dataset.select(list(range(split)))
    test            = full_dataset.select(list(range(split, len(full_dataset))))
    return DatasetDict({'train': train, 'test': test})

def _align_datasetdict_features(dataset_dict, extra_defaults=None):
    """Give every split the same columns. Hugging Face rejects DatasetDict pushes otherwise."""
    extra_defaults = extra_defaults or {}
    column_order = []
    for split in dataset_dict.values():
        for col in split.column_names:
            if col not in column_order:
                column_order.append(col)

    aligned = {}
    for name, split in dataset_dict.items():
        for col in column_order:
            if col in split.column_names:
                continue
            value = extra_defaults[col] if col in extra_defaults else (1 if col == "label" else None)
            split = split.add_column(col, [value] * len(split))
        extra_cols = [c for c in split.column_names if c not in column_order]
        aligned[name] = split.select_columns(column_order + extra_cols)
    return DatasetDict(aligned)

In [ ]:
# create folders

import os
#
if parameter_generate_dataset_sft:
    dir_hf_sft = f"{DIR_DATA_HF}/sft"
    os.makedirs(dir_hf_sft, exist_ok=True)
    print(f"Created SFT dataset folder: {dir_hf_sft}")
#
if parameter_generate_dataset_kto:
    dir_hf_kto = f"{DIR_DATA_HF}/kto"
    os.makedirs(dir_hf_kto, exist_ok=True)
    print(f"Created KTO dataset folder: {dir_hf_kto}")
#
if parameter_generate_dataset_ipo:
    dir_hf_ipo = f"{DIR_DATA_HF}/ipo"
    os.makedirs(dir_hf_ipo, exist_ok=True)
    print(f"Created IPO dataset folder: {dir_hf_ipo}")

In [ ]:
# Create HF dataset structures

#
if parameter_generate_dataset_sft:
    print("Creating HF dataset sft from JSON file:", dataset_sft)
    hf_sft = create_hf_dataset_from_json(dataset_sft, ['ground_truth'], train_ratio=parameter_ratio_train)
    print('   -> train:', len(hf_sft['train']), '; test:', len(hf_sft['test']))

    print("   Adding eval split from JSON file:", dataset_eval)
    eval_split = create_hf_dataset_from_json(dataset_eval, ['ground_truth'], split_name='eval')
    hf_sft['eval'] = eval_split['eval']
    print('   -> eval:', len(hf_sft['eval']))
    
#
if parameter_generate_dataset_kto:
    print("Creating HF dataset kto from JSON file:", dataset_kto)
    hf_kto = create_hf_dataset_from_json(dataset_kto, ['ground_truth'], label_fn=lambda r: r.get('kto_label'), train_ratio=parameter_ratio_train)
    print('   -> train:', len(hf_kto['train']), '; test:', len(hf_kto['test']))

    print("   Adding eval split from JSON file:", dataset_eval)
    # Eval is transcription ground truth, so treat missing kto_label as desirable (1).
    eval_split = create_hf_dataset_from_json(
        dataset_eval,
        ['ground_truth'],
        label_fn=lambda r: int(r['kto_label']) if r.get('kto_label') is not None else 1,
        split_name='eval',
    )
    hf_kto['eval'] = eval_split['eval']
    hf_kto = _align_datasetdict_features(hf_kto, extra_defaults={'label': 1})
    print('   -> eval:', len(hf_kto['eval']))

#
if parameter_generate_dataset_ipo:
    print("Creating HF dataset ipo from JSON file:", dataset_ipo)
    hf_ipo = create_hf_dataset_from_json(dataset_ipo, ['ipo_transcript_winner', 'ipo_transcript_loser'], label_fn=lambda r: r.get('ipo_label'), train_ratio=parameter_ratio_train)
    print('   -> train:', len(hf_ipo['train']), '; test:', len(hf_ipo['test']))

    print("   Adding eval split from JSON file:", dataset_eval)
    # Eval is transcription ground truth, so treat missing ipo_label as desirable (1).
    eval_split = create_hf_dataset_from_json(
        dataset_eval,
        ['ipo_transcript_winner', 'ipo_transcript_loser'],
        label_fn=lambda r: int(r['ipo_label']) if r.get('ipo_label') is not None else 1,
        split_name='eval',
    )
    hf_ipo['eval'] = eval_split['eval']
    hf_ipo = _align_datasetdict_features(hf_ipo, extra_defaults={'label': 1})
    print('   -> eval:', len(hf_ipo['eval']))

### Extract features

#### Helpers

In [ ]:
# define function to prepare audio and text for Whisper model

import numpy as np
import soundfile as sf
from transformers import WhisperProcessor

#NUM_PROC = min(8, max(1, (os.cpu_count() or 1) - 1))  # Use one fewer worker than cores to avoid saturating the system.
NUM_PROC = None  # None = run in main process (avoids multiprocessing crash in Jupyter)
print(f"Using {NUM_PROC} parallel process(es) for dataset preparation")

# Load the processor used for tokenization and feature extraction
print("Loading processor...")
hf_processor = WhisperProcessor.from_pretrained(
    parameter_base_model,
    language        = "it",
    task            = "transcribe",
    force_download  = False,
    cache_dir       = None,
)
print("Processor loaded successfully.")

prepare_stats = {"processed": 0, "kept": 0, "skipped": 0, "errors": 0}

#
def _not_none(example):
    return example is not None
#
def _prepare_audio_and_text(example, text_field):
    prepare_stats["processed"] += 1
    sample_idx = prepare_stats["processed"]
    audio_path = example.get("audio")
    text = example.get(text_field)

    #print(f"[prepare {sample_idx}] start -> {audio_path}")

    if not audio_path:
        prepare_stats["skipped"] += 1
        print(f"[_prepare_audio_and_text] {sample_idx} skipped: missing audio path")
        return None

    if not isinstance(text, str) or not text.strip():
        prepare_stats["skipped"] += 1
        print(f"[_prepare_audio_and_text] {sample_idx} skipped: missing text for {audio_path}")
        return None

    try:
        audio_array, sampling_rate = sf.read(audio_path)

        if isinstance(audio_array, (list, tuple)):
            audio_array = np.array(audio_array)

        if audio_array.ndim > 1:
            audio_array = audio_array.mean(axis=1)

        if len(audio_array) == 0 or sampling_rate is None or sampling_rate <= 0:
            prepare_stats["skipped"] += 1
            print(f"[_prepare_audio_and_text] {sample_idx} skipped: empty/invalid audio for {audio_path}")
            return None

        labels = hf_processor.tokenizer(
            text.strip(),
            add_special_tokens=True,
            verbose=False,
        ).input_ids

        if not labels:
            prepare_stats["skipped"] += 1
            print(f"[_prepare_audio_and_text] {sample_idx} skipped: empty tokenized labels for {audio_path}")
            return None

        input_features = hf_processor.feature_extractor(
            audio_array,
            sampling_rate=sampling_rate,
            return_attention_mask=False,
            verbose=False,
        ).input_features[0]

        example["labels"]           = labels
        example["input_features"]   = input_features
        example["input_length"]     = float(len(audio_array) / sampling_rate)

        prepare_stats["kept"] += 1
        
        '''print(
            f"[_prepare_audio_and_text] {sample_idx} {audio_path} | "
            f"text_len={len(text.strip())} | labels={len(labels)} | "
            f"features_shape={input_features.shape} | duration={example['input_length']:.2f}s"
        )'''

        if sample_idx % 50 == 0:
            print(
                f"[_prepare_audio_and_text] processed={prepare_stats['processed']} | "
                f"kept={prepare_stats['kept']} | skipped={prepare_stats['skipped']} | errors={prepare_stats['errors']}"
            )

        return example

    except Exception as exc:
        prepare_stats["errors"] += 1
        print(f"[_prepare_audio_and_text] {sample_idx} error for {audio_path}: {exc}")
        return None

#
def prepare_dataset_for_whisper(dataset, text_field="text"):
    return (
        dataset.map(
            _prepare_audio_and_text,
            fn_kwargs={"text_field": text_field},
            num_proc=NUM_PROC,
            desc=f"Preparing {text_field} dataset",
        )
        .filter(_not_none)
    )

#### Choose datasets

In [ ]:
# Choose datasets for which to extract features

datasets = []
if parameter_generate_dataset_sft:     datasets.append(("hf_sft",     hf_sft,    "text"))
if parameter_generate_dataset_kto:     datasets.append(("hf_kto",     hf_kto,    "text"))

 #### Extract

In [ ]:
# - Filter data
# - Tokenize transcriptions
# - Extract features

# This cells takes some time: ~4 minutes for ~6500 samples with device="cpu" and num_proc=None
# The cell might look stuck i.e. it keeps running but shows no output (e.g. at the "Filter" steps). if so:
# - check usage of (i) CPU and (ii) system RAM (e.g. on linux use commands: 'btop' and 'watch -n 1 nvidia-smi')
# - if (i) usage of (1 or more) CPU core(s) is high and/or (ii) RAM usage increases, the cell is still working correctly.
# e.g.: up to ~45 GB of RAM might be allocated for ~6500 samples

for name, ds, text_field in datasets:
    print(f"Extracting features for dataset {name}...")

    prepared_splits = {}
    for split_name in ds:
        print(f"\t {split_name} split...")
        prepared_splits[split_name] = prepare_dataset_for_whisper(
            ds[split_name],
            text_field=text_field,
        )
    reference_split = next(
        (split for split in prepared_splits.values() if len(split) > 0),
        None,
    )
    if reference_split is None:
        reference_split = next(iter(prepared_splits.values()), None)

    print(f"\t Reference split: {reference_split}")
    if reference_split is not None:
        for split_name, split in prepared_splits.items():
            if len(split) == 0 and split.features != reference_split.features:
                prepared_splits[split_name] = Dataset.from_dict(
                    {column: [] for column in reference_split.column_names},
                    features=reference_split.features,
                )
    ds.clear()

    #
    print(f"\t Finalizing splits...")
    for split_name, split in prepared_splits.items():
        ds[split_name] = split

In [ ]:
# free GPU memory

import torch

if torch.cuda.is_available():
    torch.cuda.empty_cache()

### Extract metadata

In [ ]:
# Build embedded file manifests for metadata.json

import json
import os
from pathlib import Path


def build_dataset_file_manifest(json_path, transcript_field):
    json_path = Path(json_path)

    if not json_path.exists():
        raise FileNotFoundError(f"Dataset JSON not found: {json_path}")

    with json_path.open("r", encoding="utf-8") as fh:
        rows = json.load(fh)

    if not isinstance(rows, list):
        raise TypeError(f"Expected a list of entries in {json_path}, got {type(rows).__name__}")

    manifest_rows = []
    for row in rows:
        if not isinstance(row, dict):
            continue

        audio_value = row.get("audio_data") or row.get("audio_path") or row.get("audio")
        transcript_value = row.get(transcript_field)

        if not audio_value or not isinstance(transcript_value, str):
            continue

        audio_path = Path(str(audio_value))
        try:
            remote_path = os.path.relpath(audio_path, DIR_DATA_RAW).replace(os.sep, "/")
        except ValueError:
            remote_path = audio_path.name

        manifest_rows.append([
            remote_path,
            transcript_value.strip(),  # or transcript_value if you want to keep exact spacing
        ])

    return manifest_rows

#
import json
from pathlib import Path

def _sum_dataset_length(ds):
    if "input_length" in ds.column_names:           return sum(ds["input_length"])
    if "chosen_input_length" in ds.column_names:    return sum(ds["chosen_input_length"])
    if "rejected_input_length" in ds.column_names:  return sum(ds["rejected_input_length"])
    return 0.0

def _dataset_metadata(ds, name):
    train_len   = _sum_dataset_length(ds["train"])  if "train"  in ds else 0.0
    test_len    = _sum_dataset_length(ds["test"])   if "test"   in ds else 0.0
    eval_len    = _sum_dataset_length(ds["eval"])   if "eval"   in ds else 0.0
    metadata = {
        "dataset-name":             name,
        "Nsamples_train-set":       len(ds["train"])    if "train"  in ds else 0,
        "Nsamples_test-set":        len(ds["test"])     if "test"   in ds else 0,
        "Nsamples_eval-set":        len(ds["eval"])     if "eval"   in ds else 0,
        "duration-hours_train-set": f"{train_len    / 3600:.2f}",
        "duration-hours_test-set":  f"{test_len     / 3600:.2f}",
        "duration-hours_eval-set":  f"{eval_len     / 3600:.2f}",
        "duration-hours_total":     f"{(train_len + test_len + eval_len) / 3600:.2f}",
    }
    return metadata


def _print_dataset_summary(ds, name):
    print(f"[_print_dataset_summary] Dataset summary for ds: {ds}, name: {name}")
    print(f"[_print_dataset_summary] Dataset splits: {list(ds.keys())}")
    
    split_lengths = {split_name: _sum_dataset_length(split) for split_name, split in ds.items()}
    total_seconds = sum(split_lengths.values())
    split_summary = ", ".join(
        f"{split_name}={split_length:.1f}s"
        for split_name, split_length in split_lengths.items()
    )
    print(f"{name}: {split_summary}, total={total_seconds:.1f}s")


In [ ]:
# set datasets to build

manifest_specs = []
#
if parameter_generate_dataset_sft:     manifest_specs.append((dataset_sft,    "ground_truth"))
if parameter_generate_dataset_kto:     manifest_specs.append((dataset_kto,    "ground_truth"))
if parameter_generate_dataset_ipo:     manifest_specs.append((dataset_ipo,    "ipo_transcript_winner"))  # or "ipo_transcript_loser" if you prefer

# build dataset files
for json_path, transcript_field in manifest_specs:
    files = build_dataset_file_manifest(json_path, transcript_field)
    print(f"{json_path}: {len(files)} files in embedded manifest")

# cleanup
try:
    del json, os
    del Path, json_path, transcript_field, manifest_specs, files
except Exception:
    pass

### Store metadata

In [ ]:
# Summaries

# get today's date in YYYYMMDD format
from datetime import datetime
today = datetime.now().strftime("%Y%m%d")
#print(today)

#
from pathlib import Path
metadata_files      = {}
dataset_json_paths  = {}
datasets            = []

#
if parameter_generate_dataset_sft:
    _print_dataset_summary(hf_sft, "hf_sft")
    
    # set HF repo id
    repo_sft = f"{ENV_HF_USER}/chiara_dataset-sft_{today}"

    #
    metadata_files["sft"]      = f"{dir_hf_sft}/metadata.json"
    dataset_json_paths["sft"]  = Path(dataset_sft)
    datasets.append(("sft",     hf_sft,   repo_sft))

#
if parameter_generate_dataset_kto:
    _print_dataset_summary(hf_kto, "hf_kto")
    
    # set HF repo id
    repo_kto = f"{ENV_HF_USER}/chiara_dataset-kto_{today}"

    #
    metadata_files["kto"]      = f"{dir_hf_kto}/metadata.json"
    dataset_json_paths["kto"]  = Path(dataset_kto)
    datasets.append(("kto",     hf_kto,   repo_kto))

#
if parameter_generate_dataset_ipo:
    _print_dataset_summary(hf_ipo, "hf_ipo")
    
    # set HF repo id
    repo_ipo = f"{ENV_HF_USER}/chiara_dataset-ipo_{today}"

    #
    metadata_files["ipo"]      = f"{dir_hf_ipo}/metadata.json"
    dataset_json_paths["ipo"]  = Path(dataset_ipo)
    datasets.append(("ipo",     hf_ipo,   repo_ipo))

In [ ]:
# Write metadata JSON files for each dataset. SFT metadata includes train/test and eval split statistics.

import shutil
import json
import soundfile as sf

for name, ds, repo_id in datasets:
    dataset_key         = name.lower()
    dataset_json_path   = dataset_json_paths[dataset_key]
    metadata            = _dataset_metadata(ds, repo_id)

    metadata_dir = Path(metadata_files[dataset_key]).parent
    dataset_json_target = metadata_dir / "dataset.json"
    if dataset_key == "sft":
        sft_entries     = _load_json(dataset_json_path)
        eval_entries    = _load_json(dataset_eval)
        with dataset_json_target.open("w", encoding="utf-8") as dataset_file:
            json.dump(sft_entries + eval_entries, dataset_file, indent=2, ensure_ascii=False)
    else:
        shutil.copy2(dataset_json_path, dataset_json_target)

    with open(metadata_files[dataset_key], "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=4, ensure_ascii=False)

        print(f"Saved metadata to {metadata_files[dataset_key]}")

### Save

#### Locally

In [ ]:
'''
if generate_dataset_sft:
    hf_sft.save_to_disk( dir_hf_sft )
    print('Saved HF SFT dataset to', dir_hf_sft)

if generate_dataset_kto:
    hf_kto.save_to_disk( dir_hf_kto )
    print('Saved HF KTO dataset to', dir_hf_kto)

if generate_dataset_ipo:
    hf_ipo.save_to_disk( dir_hf_ipo )
    print('Saved HF IPO dataset to', dir_hf_ipo)
'''

#### Push to HF hub

In [ ]:
# Push datasets and metadata to Hugging Face Hub
# Install Git LFS: https://github.com/git-lfs/git-lfs/blob/main/INSTALLING.md

# It takes ~5m to push ~10'000 samples (~15 GB) to Hugging Face Hub, depending on your internet connection speed
 
from datasets import DatasetDict
from huggingface_hub import HfApi
api = HfApi()

def _align_datasetdict_features(dataset_dict, extra_defaults=None):
    extra_defaults = extra_defaults or {}
    column_order = []
    for split in dataset_dict.values():
        for col in split.column_names:
            if col not in column_order:
                column_order.append(col)
    aligned = {}
    for name, split in dataset_dict.items():
        for col in column_order:
            if col in split.column_names:
                continue
            value = extra_defaults[col] if col in extra_defaults else (1 if col == "label" else None)
            split = split.add_column(col, [value] * len(split))
        extra_cols = [c for c in split.column_names if c not in column_order]
        aligned[name] = split.select_columns(column_order + extra_cols)
    return DatasetDict(aligned)

if ENV_HF_TOKEN and ENV_HF_USER:
    print('Pushing datasets to HF Hub under user:', ENV_HF_USER)
    try:
        # dataset-sft
        if parameter_generate_dataset_sft:
            print(f"Pushing dataset: {repo_sft}")
            #
            hf_sft.push_to_hub(f"{repo_sft}", token=ENV_HF_TOKEN, private=True)
            #
            api.upload_file(
                path_or_fileobj = f"{dir_hf_sft}/metadata.json",
                path_in_repo    = 'metadata.json',
                repo_id         = f"{repo_sft}",
                repo_type       = 'dataset',
                commit_message  = 'Add metadata.json'
            )
            #
            api.upload_file(
                path_or_fileobj = f"{dir_hf_sft}/dataset.json",
                path_in_repo    = 'data.json',
                repo_id         = f"{repo_sft}",
                repo_type       = 'dataset',
                commit_message  = 'Add data.json'
            )

        # dataset-kto
        if parameter_generate_dataset_kto:
            print(f"Pushing dataset: {repo_kto}")
            #
            hf_kto = _align_datasetdict_features(hf_kto, extra_defaults={'label': 1})
            hf_kto.push_to_hub(f"{repo_kto}", token=ENV_HF_TOKEN, private=True)
            #
            api.upload_file(
                path_or_fileobj = f"{dir_hf_kto}/metadata.json",
                path_in_repo    = 'metadata.json',
                repo_id         = f"{repo_kto}",
                repo_type       = 'dataset',
                commit_message  = 'Add metadata.json'
            )
            #
            api.upload_file(
                path_or_fileobj = f"{dir_hf_kto}/dataset.json",
                path_in_repo    = 'data.json',
                repo_id         = f"{repo_kto}",
                repo_type       = 'dataset',
                commit_message  = 'Add data.json'
            )

        # dataset-ipo
        if parameter_generate_dataset_ipo:
            print(f"Pushing dataset: {repo_ipo}")
            #
            hf_ipo = _align_datasetdict_features(hf_ipo, extra_defaults={'label': 1})
            hf_ipo.push_to_hub(f"{repo_ipo}", token=ENV_HF_TOKEN, private=True)
            #
            api.upload_file(
                path_or_fileobj = f"{dir_hf_ipo}/metadata.json",
                path_in_repo    = 'metadata.json',
                repo_id         = f"{repo_ipo}",
                repo_type       = 'dataset',
                commit_message  = 'Add metadata.json'
            )
            #
            api.upload_file(
                path_or_fileobj = f"{dir_hf_ipo}/dataset.json",
                path_in_repo    = 'data.json',
                repo_id         = f"{repo_ipo}",
                repo_type       = 'dataset',
                commit_message  = 'Add data.json'
            )
        
        # dataset-eval (only json file)
        repo_eval = f"luca-r/chiara_dataset-eval_{today}"
        print(f"Pushing dataset-eval: {repo_eval}")
        api.create_repo(
            repo_id=repo_eval,
            repo_type='dataset',
            private=True,
            exist_ok=True,
            token=ENV_HF_TOKEN,
        )
        api.upload_file(
            path_or_fileobj=dataset_eval,
            path_in_repo='data.json',
            repo_id=repo_eval,
            repo_type='dataset',
            commit_message='Add dataset-eval data.json',
            token=ENV_HF_TOKEN,
        )

        print('Push complete.')
    except Exception as e:
        print('Push failed:', e)
else:
    print('HF token or user not found; skipping push. Set HF_HUB_TOKEN and HF_HUB_USER to enable push.')

# Cleanup

In [ ]:
'''
try:
    del hf_sft
except NameError:
    pass
'''